# CSE 25 - Music Recommendation System

**Group Members:** Ovi Joshi, Jackson Horman, Johanna Sanchez

**Professor:** Minnes


## Project Summary

This project builds a content-based music recommendation system trained on Spotify audio features. 
Rather than relying on user listening history or popularity, our system analyzes the actual sound 
characteristics of a song such as danceability, energy, tempo, loudness, and valence to find 
musically similar tracks.

We train a supervised neural network on these audio features to classify songs by genre. 
Once trained, we extract the model's internal learned representations (embeddings) and use 
cosine similarity to compare songs and return the most similar recommendations.

The dataset consists of over 114,000 songs across 125 genres sourced from Kaggle's Spotify 
Tracks Dataset. We will evaluate our model using accuracy, precision, recall, and F1-score, 
and compare it against a Logistic Regression baseline.

## 1: Importing Packages

In [4]:
# core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("All packages imported successfully")

All packages imported successfully


## 2: Load the Data
We define a 'load_data' function that reads the Spotify dataset from a CSV file and prepares it for use in our model. The function does three things:


-Reads the CSV file into a pandas dataframe

-Drops any rows with missing values in our audio feature columns

-Removes duplicate songs so we don't train on the same song twice


In [6]:
def load_data(filepath):
    # Read the CSV file into a pandas dataframe
    df = pd.read_csv(filepath)

    # These are the audio features we will use to train our model
    feature_cols = [
        'danceability',     # How suitable a track is for dancing
        'energy',           # Intensity and activity of the track
        'loudness',         # Overall loudness in decibels
        'speechiness',      # Presence of spoken words
        'acousticness',     # How acoustic the track is
        'instrumentalness', # Whether a track has no vocals
        'liveness',         # Presence of a live audience
        'valence',          # Musical positiveness of the track
        'tempo'             # Speed of the track in BPM
    ]

    # Remove rows where any feature value or genre is missing
    df = df.dropna(subset=feature_cols + ['track_genre'])

    # Remove duplicate songs keeping only the first occurrence
    df = df.drop_duplicates(subset='track_id')

    # Print a summary so we can verify the data loaded correctly
    print(f"Dataset shape: {df.shape}")
    print(f"Number of genres: {df['track_genre'].nunique()}")
    print(f"Sample:\n{df[feature_cols].head()}")

    return df, feature_cols

# Call the function with the path to our dataset
df, feature_cols = load_data('data/dataset.csv')

Dataset shape: (89741, 21)
Number of genres: 113
Sample:
   danceability  energy  loudness  speechiness  acousticness  \
0         0.676  0.4610    -6.746       0.1430        0.0322   
1         0.420  0.1660   -17.235       0.0763        0.9240   
2         0.438  0.3590    -9.734       0.0557        0.2100   
3         0.266  0.0596   -18.515       0.0363        0.9050   
4         0.618  0.4430    -9.681       0.0526        0.4690   

   instrumentalness  liveness  valence    tempo  
0          0.000001    0.3580    0.715   87.917  
1          0.000006    0.1010    0.267   77.489  
2          0.000000    0.1170    0.120   76.332  
3          0.000071    0.1320    0.143  181.740  
4          0.000000    0.0829    0.167  119.949  
